# Appendix 3.3: Instrumental Variable

This notebook reviews an additional example of shelving estimating functions with instrumental variables. Data for this example comes from a pre-processed version of the Oregon Health Insurance Experiment. In particular, the data is restricted to complete-cases among households with more than 2 participants. Further a random sample of 200 households was taken to further highlight the impacts of ignoring clustering.

In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import delicatessen as deli
from delicatessen import MEstimator
from delicatessen.estimating_equations import ee_2sls
from delicatessen.utilities import aggregate_efuncs

print("Versions")
print("NumPy:       ", np.__version__)
print("SciPy:       ", sp.__version__)
print("Pandas:      ", pd.__version__)
print("Delicatessen:", deli.__version__)

Versions
NumPy:        2.3.5
SciPy:        1.16.3
Pandas:       2.3.3
Delicatessen: 4.3


In [2]:
# Setting up the data
d = pd.read_csv("data/ohie_subsample.csv")

In [3]:
# Defining the estimating function
def psi_i(theta):
    # Individual-level estimating function
    return ee_2sls(theta,              # Input parameters for Two-Stage Least Squares
                   Z=d[['Z', ]],       # Instrumental variable (lottery)
                   A=d['A'],           # Action variable (accepted application)
                   y=d['Y'],           # Outcome of interest
                   W=d[['I', 'W']])    # Additional covariates in both model stages

def psi_g(theta):
    # Household-aggregated estimating functions
    return aggregate_efuncs(est_funcs=psi_i(theta),
                            group=d['hid'])

In [4]:
estr1 = MEstimator(psi_i, init=[0., ]*3*2)
estr1.estimate()
estr1.print_results(subset=[0, ], decimals=3)

              Estimation Method: M-estimator
--------------------------------------------------------------
No. Observations:         401 | No. Parameters:              6
Solving algorithm:         lm | Max Iterations:           5000
Solving tolerance:      1e-09 | Allow P-Inverse:             1
Derivative Method:     approx | Deriv Approx:            1e-09
Small N Correction:      None | Distribution:           Z-stat
   Theta   StdErr  Z-score      LCL      UCL  P-value  S-value 
--------------------------------------------------------------
   0.186    0.229    0.810   -0.264    0.636    0.418    1.258 


In [5]:
estr2 = MEstimator(psi_g, init=[0., ]*3*2)
estr2.estimate()
estr2.print_results(subset=[0, ], decimals=3)

              Estimation Method: M-estimator
--------------------------------------------------------------
No. Observations:         200 | No. Parameters:              6
Solving algorithm:         lm | Max Iterations:           5000
Solving tolerance:      1e-09 | Allow P-Inverse:             1
Derivative Method:     approx | Deriv Approx:            1e-09
Small N Correction:      None | Distribution:           Z-stat
   Theta   StdErr  Z-score      LCL      UCL  P-value  S-value 
--------------------------------------------------------------
   0.186    0.242    0.769   -0.288    0.660    0.442    1.178 
